# Clusterização de municípios com KMeans

Notebook reorganizado para publicação no GitHub, com foco em **reprodutibilidade**, **leitura clara** e **proteção dos dados da pesquisa**.

## O que este notebook faz
1. Carrega a base PCA por município.
2. Ajusta o modelo **KMeans**.
3. Gera os clusters e salva os artefatos principais.
4. Faz o merge dos clusters no painel município-ano.
5. Produz visualizações no espaço PCA, nas features agregadas e no espaço geográfico.

## Observação importante
Os **bancos de dados não estão incluídos** no repositório nesta etapa, para evitar exposição indevida antes da publicação da pesquisa.

## Estrutura esperada de pastas
```text
data/
├── aux/
│   └── municipios_sp_coordenadas.csv
├── features/
│   ├── X_pca_municipio.parquet
│   └── features_municipio_mean_std_slope.parquet
├── processed/
│   └── painel_municipio_ano_2007_2021.parquet
├── outputs/
└── models/
```


In [ ]:
# Configuração geral

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Diretórios
DATA_DIR = Path("data")
FEAT_DIR = DATA_DIR / "features"
PROC_DIR = DATA_DIR / "processed"
AUX_DIR = DATA_DIR / "aux"
OUT_DIR = DATA_DIR / "outputs"
MODEL_DIR = DATA_DIR / "models"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Arquivos de entrada
IN_XPCA = FEAT_DIR / "X_pca_municipio.parquet"
IN_PANEL = PROC_DIR / "painel_municipio_ano_2007_2021.parquet"
IN_FEATS = FEAT_DIR / "features_municipio_mean_std_slope.parquet"
COORD_FILE = AUX_DIR / "municipios_sp_coordenadas.csv"

# Arquivos de saída
OUT_CLUSTER_MUN = OUT_DIR / "cluster_municipio_kmeans.parquet"
OUT_PANEL_CLUSTER = OUT_DIR / "painel_municipio_ano_cluster.parquet"
OUT_KMEANS_MODEL = MODEL_DIR / "kmeans_municipio.joblib"

# Parâmetros do modelo
K = 5
RANDOM_STATE = 42
N_INIT = 30

print("Notebook configurado.")
print(f"K = {K}")
print(f"Entrada PCA: {IN_XPCA}")


In [ ]:
# Funções auxiliares

def validar_arquivo(caminho: Path) -> None:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}\n"
            "Como os dados não serão publicados agora, mantenha apenas a estrutura "
            "de pastas no GitHub e adicione os arquivos localmente quando for executar."
        )


def padronizar_municipio(df: pd.DataFrame, coluna: str = "Municipio") -> pd.DataFrame:
    df = df.copy()
    df[coluna] = df[coluna].astype(str).str.strip()
    return df


## 1. Carga da base PCA e treinamento do KMeans

In [ ]:
# Carregar matriz PCA por município

validar_arquivo(IN_XPCA)

df_xpca = pd.read_parquet(IN_XPCA)
df_xpca = padronizar_municipio(df_xpca, "Municipio")

X = df_xpca.drop(columns=["Municipio"])

display(df_xpca.head())
print(df_xpca.shape)


In [ ]:
# Ajustar KMeans e gerar labels

kmeans = KMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
)

labels = kmeans.fit_predict(X)

df_cluster = df_xpca[["Municipio"]].copy()
df_cluster["cluster"] = labels

df_cluster.to_parquet(OUT_CLUSTER_MUN, index=False)
joblib.dump(kmeans, OUT_KMEANS_MODEL)

display(df_cluster.head())
display(df_cluster["cluster"].value_counts().sort_index())


## 2. Merge dos clusters no painel município-ano

In [ ]:
# Incorporar cluster ao painel principal

validar_arquivo(IN_PANEL)

df_panel = pd.read_parquet(IN_PANEL)
df_panel = padronizar_municipio(df_panel, "Municipio")

df_panel_cluster = df_panel.merge(df_cluster, on="Municipio", how="left")
df_panel_cluster.to_parquet(OUT_PANEL_CLUSTER, index=False)

display(df_panel_cluster.head())
print(df_panel_cluster.shape)


## 3. Visualizações básicas dos clusters no espaço PCA

In [ ]:
# Distribuição dos clusters

counts = df_cluster["cluster"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Distribuição dos clusters")
plt.xlabel("Cluster")
plt.ylabel("Quantidade de municípios")
plt.grid(axis="y", alpha=0.3)
plt.show()

counts


In [ ]:
# Dispersão PC1 x PC2 por cluster

df_vis = df_xpca.merge(df_cluster, on="Municipio", how="left")
pc_cols = [c for c in df_xpca.columns if c.startswith("PC")]

plt.figure(figsize=(8, 6))
for c in sorted(df_vis["cluster"].dropna().unique()):
    mask = df_vis["cluster"] == c
    plt.scatter(df_vis.loc[mask, "PC1"], df_vis.loc[mask, "PC2"], s=12, alpha=0.7, label=f"Cluster {c}")

plt.title("Clusters no espaço PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Centróides do KMeans no plano PC1 x PC2

centroids = kmeans.cluster_centers_

plt.figure(figsize=(8, 6))
plt.scatter(df_vis["PC1"], df_vis["PC2"], s=8, alpha=0.2)

plt.scatter(centroids[:, 0], centroids[:, 1], s=200, marker="X", edgecolor="black")
for i, (x, y) in enumerate(centroids[:, :2]):
    plt.text(x, y, f"C{i}", fontsize=10)

plt.title("Centróides dos clusters no espaço PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Perfil médio dos clusters nas componentes principais

df_pc_profile = df_vis.groupby("cluster")[pc_cols].mean()

plt.figure(figsize=(11, 4))
for c in df_pc_profile.index:
    plt.plot(df_pc_profile.columns, df_pc_profile.loc[c].values, marker="o", label=f"Cluster {c}")

plt.title("Perfil médio dos clusters nas PCs")
plt.xlabel("Componentes principais")
plt.ylabel("Média")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

df_pc_profile


In [ ]:
# Distância entre centróides no espaço PCA

D = np.sqrt(((centroids[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))

plt.figure(figsize=(6, 5))
plt.imshow(D, aspect="auto")
plt.title("Distância entre centróides")
plt.xlabel("Cluster")
plt.ylabel("Cluster")
plt.xticks(range(D.shape[0]), [f"C{i}" for i in range(D.shape[0])])
plt.yticks(range(D.shape[0]), [f"C{i}" for i in range(D.shape[0])])
plt.colorbar(label="Distância euclidiana")
plt.show()

pd.DataFrame(D, index=[f"C{i}" for i in range(D.shape[0])], columns=[f"C{i}" for i in range(D.shape[0])])


## 4. Interpretação com features agregadas

In [ ]:
# Carregar features agregadas por município

validar_arquivo(IN_FEATS)

df_feats = pd.read_parquet(IN_FEATS)
df_feats = padronizar_municipio(df_feats, "Municipio")

df_feats_cluster = df_feats.merge(df_cluster, on="Municipio", how="left")
feature_cols = [c for c in df_feats_cluster.columns if c not in ["Municipio", "cluster"]]

display(df_feats_cluster.head())
print(f"Total de features agregadas: {len(feature_cols)}")


In [ ]:
# Perfil médio dos clusters nas features originais agregadas

profile_feats = df_feats_cluster.groupby("cluster")[feature_cols].mean()

display(profile_feats.head())


In [ ]:
# Ranking simples das variáveis que mais diferenciam clusters

between_var = profile_feats.var(axis=0)
total_var = df_feats_cluster[feature_cols].var(axis=0)
score = (between_var / total_var.replace(0, np.nan)).sort_values(ascending=False)

top_vars = score.dropna()
display(top_vars.head(15))


In [ ]:
# Boxplots das variáveis mais discriminantes

top_features = list(top_vars.index[:6])

for col in top_features:
    plt.figure(figsize=(10, 4))
    data = [
        df_feats_cluster.loc[df_feats_cluster["cluster"] == c, col].dropna().values
        for c in sorted(df_feats_cluster["cluster"].dropna().unique())
    ]

    plt.boxplot(data, tick_labels=[f"C{c}" for c in sorted(df_feats_cluster["cluster"].dropna().unique())])
    plt.title(f"Distribuição de {col} por cluster")
    plt.xlabel("Cluster")
    plt.ylabel(col)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


In [ ]:
# Violin plots com corte por quantis para reduzir distorção por outliers

clusters = sorted(df_feats_cluster["cluster"].dropna().unique())

for col in top_features:
    plt.figure(figsize=(10, 4))

    data = []
    for c in clusters:
        vals = df_feats_cluster.loc[df_feats_cluster["cluster"] == c, col].dropna().values
        if len(vals) > 0:
            q01, q99 = np.quantile(vals, [0.01, 0.99])
            vals = vals[(vals >= q01) & (vals <= q99)]
        data.append(vals)

    plt.violinplot(data, showmeans=True, showmedians=True)
    plt.xticks(range(1, len(clusters) + 1), [f"C{c}" for c in clusters])
    plt.title(f"Distribuição ajustada de {col} por cluster")
    plt.xlabel("Cluster")
    plt.ylabel(col)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


## 5. Visualização geográfica dos clusters

In [ ]:
# Carregar coordenadas dos municípios

validar_arquivo(COORD_FILE)

df_coord = pd.read_csv(COORD_FILE)
df_coord = padronizar_municipio(df_coord, "Municipio")

# Ajuste opcional de nomes de colunas, se necessário
rename_map = {}
if "lat" in df_coord.columns and "latitude" not in df_coord.columns:
    rename_map["lat"] = "latitude"
if "lon" in df_coord.columns and "longitude" not in df_coord.columns:
    rename_map["lon"] = "longitude"

if rename_map:
    df_coord = df_coord.rename(columns=rename_map)

display(df_coord.head())


In [ ]:
# Merge coordenadas + cluster

df_geo = df_cluster.merge(df_coord, on="Municipio", how="left")

print("Municípios sem coordenada:", df_geo["latitude"].isna().sum())
display(df_geo.head())


In [ ]:
# Dispersão espacial dos municípios por cluster

plt.figure(figsize=(8, 10))

for c in sorted(df_geo["cluster"].dropna().unique()):
    mask = df_geo["cluster"] == c
    plt.scatter(
        df_geo.loc[mask, "longitude"],
        df_geo.loc[mask, "latitude"],
        s=12,
        alpha=0.7,
        label=f"Cluster {c}",
    )

plt.title("Dispersão espacial dos clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Centróide geográfico dos clusters

cent_geo = df_geo.groupby("cluster")[["latitude", "longitude"]].mean()

plt.figure(figsize=(8, 10))
plt.scatter(df_geo["longitude"], df_geo["latitude"], s=8, alpha=0.2, color="gray")
plt.scatter(cent_geo["longitude"], cent_geo["latitude"], s=200, marker="X")

for c, row in cent_geo.iterrows():
    plt.text(row["longitude"], row["latitude"], f"C{c}", fontsize=10)

plt.title("Centróides geográficos dos clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, alpha=0.3)
plt.show()

cent_geo


In [ ]:
# Comparação: espaço PCA vs espaço geográfico

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for c in sorted(df_vis["cluster"].dropna().unique()):
    mask = df_vis["cluster"] == c
    axes[0].scatter(df_vis.loc[mask, "PC1"], df_vis.loc[mask, "PC2"], s=10, alpha=0.7, label=f"C{c}")

axes[0].set_title("Espaço PCA")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(True, alpha=0.3)

for c in sorted(df_geo["cluster"].dropna().unique()):
    mask = df_geo["cluster"] == c
    axes[1].scatter(df_geo.loc[mask, "longitude"], df_geo.loc[mask, "latitude"], s=10, alpha=0.7, label=f"C{c}")

axes[1].set_title("Espaço geográfico")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 5))
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## 6. Tabelas finais de apoio

In [ ]:
# Contagem de municípios por cluster

contagem_clusters = (
    df_cluster["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="n_municipios")
)

display(contagem_clusters)
print(f"Total de municípios analisados: {contagem_clusters['n_municipios'].sum()}")


In [ ]:
# Listar colunas do painel final em formato copiável

colunas = df_panel_cluster.columns.tolist()

print(f"Total de colunas: {len(colunas)}\n")
for c in colunas:
    print(f'"{c}",')
